# ⚡ VibraDistill-Edge: Kaggle Dual NVIDIA Tesla T4 GPU Training Pipeline
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code)

> **Hardware Target:** Sipeed Tang Primer 20K FPGA (Gowin GW2A-LV18PG256C8/I7 @ 100 MHz NPU, 27 MHz Onboard OSC) + Sonix SN32F407 MCU (ARM Cortex-M0 @ 60 MHz)
> **Kaggle Configuration:**
> - **Accelerator:** `GPU T4 x2` (30 GB total VRAM)
> - **Persistence:** `Variables and Files`
> - **Internet:** `On`
> **Primary Objective:** Zero-Leakage Bearing Fault Diagnosis with QAT and DKD under strict $\le 10$ KB INT8 weight budget.


## 1. Verify Dual NVIDIA Tesla T4 GPU Allocation
Detects both `cuda:0` and `cuda:1` devices.


In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version: {torch.__version__}")
num_gpus = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {num_gpus}")

for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU [{i}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute Cap: {props.major}.{props.minor}")

if num_gpus < 2:
    print("Notice: GPU T4 x2 is recommended in Kaggle Settings -> Accelerator -> GPU T4 x2.")


## 2. Setup Codebase & Auto-Link Kaggle Input Datasets
Clones repository and maps any attached Kaggle datasets (`/kaggle/input/...`) to `data/`.


In [ ]:
import os, sys, shutil, subprocess

WORKSPACE_DIR = "/kaggle/working/VIbraDistill" if os.path.exists("/kaggle") else os.getcwd()

# 1. Explicit Candidate Paths (matching your uploaded dataset slug & URLs)
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/zok213/vibradistill-ai1",
    "/kaggle/input/datasets/zok213/vibradistill_ai1",
    "/kaggle/input/vibradistill_ai1",
    "/kaggle/input/vibradistill-ai1",
    "/kaggle/input/vibradistill-all-in-one",
]

ALL_IN_ONE_PATH = None
for cand in CANDIDATE_PATHS:
    if os.path.exists(cand) and os.path.exists(os.path.join(cand, "src")):
        ALL_IN_ONE_PATH = cand
        break

# 2. Universal Auto-Scan in /kaggle/input (finds any directory containing src and scripts)
if not ALL_IN_ONE_PATH and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs and "scripts" in dirs:
            ALL_IN_ONE_PATH = root
            break

if ALL_IN_ONE_PATH:
    print(f"✅ Đã tìm thấy Master Bundle tại: {ALL_IN_ONE_PATH}")
    os.makedirs(WORKSPACE_DIR, exist_ok=True)
    for item in os.listdir(ALL_IN_ONE_PATH):
        src_item = os.path.join(ALL_IN_ONE_PATH, item)
        dst_item = os.path.join(WORKSPACE_DIR, item)
        if item == "data":
            os.makedirs(dst_item, exist_ok=True)
            for d_name in os.listdir(src_item):
                d_src = os.path.join(src_item, d_name)
                d_dst = os.path.join(dst_item, d_name)
                if not os.path.exists(d_dst):
                    try:
                        os.symlink(d_src, d_dst)
                    except Exception:
                        shutil.copytree(d_src, d_dst)
        elif not os.path.exists(dst_item):
            if os.path.isdir(src_item):
                shutil.copytree(src_item, dst_item, dirs_exist_ok=True)
            else:
                shutil.copy2(src_item, dst_item)
    os.chdir(WORKSPACE_DIR)
elif os.path.exists("/kaggle"):
    if not os.path.exists(WORKSPACE_DIR):
        subprocess.run(["git", "clone", "https://github.com/zok213/VIbraDistill.git", WORKSPACE_DIR], check=True)
        os.chdir(WORKSPACE_DIR)
    else:
        os.chdir(WORKSPACE_DIR)
        subprocess.run(["git", "pull"], check=True)

# 3. Self-Healing Patch: Ensure scripts have latest fixes even if loaded from older Kaggle dataset
if os.path.exists("scripts/08_qat_train.py"):
    with open("scripts/08_qat_train.py", "r", encoding="utf-8") as f:
        qat_script = f.read()
    old_pointer_bug = "teacher = copy.deepcopy(folded).to(dev).eval() if False else folded.to(dev).eval()"
    if old_pointer_bug in qat_script:
        qat_script = qat_script.replace(
            old_pointer_bug + "\n    qat = QATMicro(folded.to('cpu'), ranges).to(dev)",
            "qat = QATMicro(folded.to('cpu'), ranges).to(dev)\n    teacher = copy.deepcopy(folded).to(dev).eval()"
        )
        qat_script = qat_script.replace("tot += float(loss) * len(y)", "tot += float(loss.detach()) * len(y)")
        with open("scripts/08_qat_train.py", "w", encoding="utf-8") as f:
            f.write(qat_script)
        print("🔧 Auto-healed: scripts/08_qat_train.py device pointer fix applied!")

if os.path.exists("scripts/download_all_benchmarks.py"):
    with open("scripts/download_all_benchmarks.py", "r", encoding="utf-8") as f:
        dl_script = f.read()
    if "--benchmark" not in dl_script:
        dl_script = dl_script.replace(
            'parser.add_argument("--dataset",',
            'parser.add_argument("--benchmark", default=None, help="Alias for --dataset")\n    parser.add_argument("--dataset",'
        )
        dl_script = dl_script.replace(
            'target_keys = list(DATASETS_METADATA.keys()) if args.dataset == "all" else [args.dataset]',
            'sel = args.benchmark if args.benchmark is not None else args.dataset\n    target_keys = list(DATASETS_METADATA.keys()) if sel == "all" else [sel]'
        )
        with open("scripts/download_all_benchmarks.py", "w", encoding="utf-8") as f:
            f.write(dl_script)
        print("🔧 Auto-healed: scripts/download_all_benchmarks.py --benchmark alias added!")

# 4. Safe pip install (protecting Python 3.13 pre-installed numpy)
if os.path.exists("requirements.txt"):
    with open("requirements.txt", "r", encoding="utf-8") as f:
        reqs = f.read()
    if "numpy<2.0.0" in reqs:
        reqs = reqs.replace("numpy>=1.24.0,<2.0.0", "numpy>=1.24.0")
        with open("requirements.txt", "w", encoding="utf-8") as f:
            f.write(reqs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--upgrade-strategy", "only-if-needed"], check=False)

# 5. Link individual Kaggle Input Datasets if attached separately
KAGGLE_INPUT_DIR = "/kaggle/input"
os.makedirs("data", exist_ok=True)

DATASET_MAP = {
    "vibradistill-cwru-dataset": "CWRU_Dataset",
    "cwru": "CWRU_Dataset",
    "vibradistill-mfpt-dataset": "MFPT_Dataset",
    "vibradistill-seu-dataset": "SEU_Dataset",
    "vibradistill-phm2009-dataset": "PHM2009_Gearbox_Dataset",
    "vibradistill-rotating-machine-faults": "Rotating_Machine_Faults_Dataset",
    "vibradistill-xjtu-sy-dataset": "XJTU-SY_Dataset",
    "vibradistill-pronostia-femto-dataset": "PRONOSTIA_FEMTO_Dataset"
}

if os.path.exists(KAGGLE_INPUT_DIR):
    for k_name in os.listdir(KAGGLE_INPUT_DIR):
        k_lower = k_name.lower()
        for pattern, local_target in DATASET_MAP.items():
            if pattern in k_lower:
                src = os.path.join(KAGGLE_INPUT_DIR, k_name)
                sub = [os.path.join(src, f) for f in os.listdir(src) if os.path.isdir(os.path.join(src, f))]
                actual_src = sub[0] if (len(sub) == 1 and local_target.lower() in sub[0].lower()) else src
                dst = os.path.join("data", local_target)
                if not os.path.exists(dst):
                    try:
                        os.symlink(actual_src, dst)
                        print(f"Linked: {src} -> {dst}")
                    except Exception:
                        shutil.copytree(actual_src, dst)
                        print(f"Copied: {src} -> {dst}")

# 6. Verify CWRU dataset presence (check for actual .mat files in Train_7mil)
cwru_train = "data/CWRU_Dataset/Train_7mil"
if not os.path.exists(cwru_train) or len(os.listdir(cwru_train)) == 0:
    print("CWRU dataset not found in local workspace; downloading directly...")
    subprocess.run([sys.executable, "scripts/download_all_benchmarks.py", "--dataset", "cwru"], check=True)
else:
    print(f"✅ CWRU dataset ready ({len(os.listdir(cwru_train))} files in Train_7mil)")

print(f"\n[STATUS] Active Working Directory: {os.getcwd()}")
if os.path.exists("data"):
    print(f"[STATUS] Mounted Datasets in data/: {sorted(os.listdir('data'))}")


## 3. Verify Bit-True Causal Hardware DSP Pipeline


In [ ]:
!python scripts/01_verify_dsp.py


## 4. Concurrent Dual-GPU Parallel Orchestrator
Leverages Kaggle Dual T4 GPUs simultaneously:
- **GPU 0 (`cuda:0`)**: Trains Teacher 1D-ResNet Oracle (~150k params)
- **GPU 1 (`cuda:1`)**: Concurrently runs Multi-Model Baseline Benchmark (`SingleKernelCNN`, `GRU`, `BiLSTM`, `MLP`)


In [ ]:
import subprocess, time
from concurrent.futures import ThreadPoolExecutor

print("=" * 75)
print("LAUNCHING DUAL-GPU CONCURRENT WORKLOADS")
print("=" * 75)

def run_task(cmd, gpu_name):
    t0 = time.time()
    print(f"[{gpu_name} START] {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    dur = time.time() - t0
    if proc.returncode == 0:
        print(f"[{gpu_name} FINISHED] in {dur:.1f}s")
        lines = [line for line in proc.stdout.splitlines() if line.strip()]
        for l in lines[-12:]:
            print(f"  [{gpu_name}] {l}")
    else:
        print(f"[{gpu_name} FAILED] in {dur:.1f}s")
        if proc.stderr:
            print(f"[{gpu_name} STDERR]\n{proc.stderr}")
        if proc.stdout:
            print(f"[{gpu_name} STDOUT]\n{proc.stdout}")
        raise RuntimeError(f"{gpu_name} failed with return code {proc.returncode}")
    return proc.stdout

num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(run_task, "python scripts/02_train_teacher.py --epochs 30 --batch_size 64 --device cuda:0", "GPU 0")
        f1 = ex.submit(run_task, "python scripts/100_runs_ablation_benchmark.py --num_runs 5 --device cuda:1", "GPU 1")
        print("Both GPU 0 and GPU 1 are active in parallel...")
        f0.result()
        f1.result()
else:
    dev = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(f"Running sequentially on {dev}...")
    run_task(f"python scripts/02_train_teacher.py --epochs 30 --batch_size 64 --device {dev}", "GPU 0")
    run_task(f"python scripts/100_runs_ablation_benchmark.py --num_runs 5 --device {dev}", "GPU 0")


## 5. Decoupled Knowledge Distillation (DKD) & Evidential Learning into Student (`VibraDistillMicro`)
Transfers semantic dark knowledge into the ultra-compact 8,677 parameter model.


In [ ]:
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
!python scripts/03_train_student_dkd.py --epochs 35 --batch_size 64 --temperature 5.0 --alpha 1.0 --beta 2.0 --edl_weight 0.5 --device {dev}


## 6. Strict Zero-Leakage Benchmark Evaluation
Evaluates generalization on physically unseen bearing fault datasets (`14mil` and `21mil`).


In [ ]:
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
!python scripts/04_evaluate_zero_leakage.py --device {dev}


## 7. Quantization-Aware Training (QAT) with Straight-Through Estimators (STE)
Eliminates post-training quantization collapse on INT8 edge silicon.


In [ ]:
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
!python scripts/08_qat_train.py --device {dev} --epochs 15 --batch_size 64 --lr 0.0005


## 8. Verify Bit-True INT8 Parity on Gowin NPU 12-Way Hardware Emulator
Confirms exact decision agreement between float simulation and bit-true hardware emulator.


In [ ]:
!python scripts/07_verify_int8_parity.py --checkpoint checkpoints/student_qat/best_qat.pt


## 9. Hardware Silicon Export for Sipeed Tang Primer 20K & Sonix SN32F407
Generates:
- 6-Bank Gowin Primer 20K (GW2A-LV18PG256C8/I7) BSRAM `.mi` hex ROM files for 12-way NPU (13.0% BSRAM utilization)
- ANSI C headers `vibradistill_weights.h` and model headers for Sonix SN32F407 MCU (< 512 bytes SRAM, 26.5% Flash)
- Ready for Gowin EDA synthesis with 27 MHz onboard crystal oscillator rPLL IP!


In [ ]:
!python scripts/05_quantize_and_export.py --checkpoint checkpoints/student_qat/best_qat.pt


## 10. Multi-Dataset Comprehensive Benchmark & OOD Evaluation
Discovers all attached datasets (CWRU, MFPT, PRONOSTIA, SEU, PHM2009, Rotating Machine) and evaluates in-domain, zero-shot transfer, and epistemic OOD rejection.


In [ ]:
# Run comprehensive multi-dataset benchmark across all mounted datasets
!python scripts/09_multi_dataset_benchmark.py

# If XJTU-SY dataset is attached, also run adaptive conformal RUL prognostics:
import os, subprocess, sys
if os.path.exists("data/XJTU-SY_Dataset"):
    subprocess.run([sys.executable, "scripts/06_train_rul_conformal.py", "--epochs", "10", "--condition", "35Hz12kN"], check=True)


## 11. Package Output Artifacts for Kaggle 1-Click Download
Creates an all-in-one ZIP archive in `/kaggle/working/` containing checkpoints, ROM files, C headers, and benchmark JSON logs.


In [ ]:
import zipfile

out_zip = "/kaggle/working/vibradistill_kaggle_artifacts.zip" if os.path.exists("/kaggle") else "vibradistill_kaggle_artifacts.zip"
print(f"Creating download archive: {out_zip}...")

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for target in ['checkpoints', 'experiments', 'embedded/fpga_gowin/roms', 'embedded/mcu_sonix/app']:
        if os.path.exists(target):
            for root, _, files in os.walk(target):
                for f in files:
                    fp = os.path.join(root, f)
                    arc = os.path.relpath(fp, '.')
                    z.write(fp, arc)

size_mb = os.path.getsize(out_zip) / (1024 * 1024)
print(f"Successfully packaged {size_mb:.2f} MB!")
print("You can download this archive directly from the Kaggle Notebook Output tab on the right sidebar.")
